# T-type taxonomy

Register the shared Tasic 2018 VISp hierarchy with plotting colors. This notebook writes taxonomy metadata only; dataset assignments belong in cohort ETLs.

In [1]:
from pathlib import Path

import pandas as pd
import polars as pl

from connects_common_connectivity.config import get_settings
from connects_common_connectivity.io import write_models
from connects_common_connectivity.models import (
    AlgorithmRun,
    Cluster,
    ClusterHierarchy,
    HierarchyCategory,
)

In [2]:
REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
INPUT_PATH = REPO_ROOT / "data/T-type colors Tasic/anno.feather"
OUTPUT_ROOT = Path(get_settings().output_root)

HIERARCHY_ID = "tasic_2018_visp_taxonomy"
RUN_ID = "tasic_2018_visp_clustering"
ROOT_ID = "cell"
assert INPUT_PATH.is_file(), f"Missing local input: {INPUT_PATH}"

## Shape hierarchy and colors

In [3]:
source = pd.read_feather(INPUT_PATH)
required_columns = {
    "class_label", "subclass_label", "cluster_label",
    "class_color", "subclass_color", "cluster_color",
}
assert required_columns <= set(source.columns)
assert source[list(required_columns)].notna().all().all()

# Each child must have one parent before labels can become CCM hierarchy nodes.
for child, parent in (("cluster_label", "subclass_label"), ("subclass_label", "class_label")):
    parent_counts = source.groupby(child)[parent].nunique()
    assert parent_counts.le(1).all(), f"{child} maps to multiple {parent} values"

class_labels = sorted(source["class_label"].unique())
subclass_labels = sorted(source["subclass_label"].unique())
cluster_labels = sorted(source["cluster_label"].unique())
subclass_to_class = dict(zip(source["subclass_label"], source["class_label"]))
cluster_to_subclass = dict(zip(source["cluster_label"], source["subclass_label"]))
class_to_subclasses = {
    label: sorted(node for node, parent in subclass_to_class.items() if parent == label)
    for label in class_labels
}
subclass_to_clusters = {
    label: sorted(node for node, parent in cluster_to_subclass.items() if parent == label)
    for label in subclass_labels
}

class_colors = dict(zip(source["class_label"], source["class_color"]))
subclass_colors = dict(zip(source["subclass_label"], source["subclass_color"]))
cluster_colors = dict(zip(source["cluster_label"], source["cluster_color"]))

# The source carries two historical Non-Neuronal colors; retain the established Tasic palette value.
non_neuronal_colors = set(source.loc[source["class_label"] == "Non-Neuronal", "class_color"])
assert non_neuronal_colors == {"#808285", "#8D1800"}
class_colors["Non-Neuronal"] = "#8D1800"
assert (len(class_labels), len(subclass_labels), len(cluster_labels)) == (3, 23, 111)

In [4]:
categories = [
    HierarchyCategory(id="cluster", hierarchy_id=HIERARCHY_ID, level=0, description="Tasic leaf T-type."),
    HierarchyCategory(id="subclass", hierarchy_id=HIERARCHY_ID, level=1, description="Tasic transcriptomic subclass."),
    HierarchyCategory(id="class", hierarchy_id=HIERARCHY_ID, level=2, description="Tasic transcriptomic class."),
    HierarchyCategory(id="major_class", hierarchy_id=HIERARCHY_ID, level=3, description="Synthetic taxonomy root."),
]
run = AlgorithmRun(
    id=RUN_ID,
    algorithm_name="Tasic 2018 VISp transcriptomic taxonomy",
    algorithm_version="2018",
)

# Cluster.level measures depth from the root; HierarchyCategory.level orders detail from leaves upward.
clusters = [
    Cluster(
        id=ROOT_ID, hierarchy_id=HIERARCHY_ID, children=class_labels,
        level=0, hex_color="#000000", hierarchy_category="major_class",
    )
]
clusters.extend(
    Cluster(
        id=label, hierarchy_id=HIERARCHY_ID, parent=ROOT_ID,
        children=class_to_subclasses[label], level=1,
        hex_color=class_colors[label], hierarchy_category="class",
    )
    for label in class_labels
)
clusters.extend(
    Cluster(
        id=label, hierarchy_id=HIERARCHY_ID, parent=subclass_to_class[label],
        children=subclass_to_clusters[label], level=2,
        hex_color=subclass_colors[label], hierarchy_category="subclass",
    )
    for label in subclass_labels
)
clusters.extend(
    Cluster(
        id=label, hierarchy_id=HIERARCHY_ID, parent=cluster_to_subclass[label],
        children=[], level=3, hex_color=cluster_colors[label],
        hierarchy_category="cluster",
    )
    for label in cluster_labels
)
hierarchy = ClusterHierarchy(
    id=HIERARCHY_ID, run=RUN_ID, root=ROOT_ID,
    clusters=[cluster.id for cluster in clusters],
)
assert len(clusters) == 138
assert all(cluster.hex_color is not None for cluster in clusters)

## Write shared taxonomy

In [5]:
print(write_models([run], output_root=OUTPUT_ROOT))
print(write_models(categories, output_root=OUTPUT_ROOT))
print(write_models(clusters, output_root=OUTPUT_ROOT))
print(write_models([hierarchy], output_root=OUTPUT_ROOT))

WrittenResult(class_name='AlgorithmRun', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/algorithmrun'), mode='merge_scoped', predicates=('target."id" = source."id"',), rows_written=1)
WrittenResult(class_name='HierarchyCategory', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/hierarchycategory'), mode='merge_scoped', predicates=('target."hierarchy_id" = source."hierarchy_id" AND target."id" = source."id" AND target."hierarchy_id" IN (\'tasic_2018_visp_taxonomy\')',), rows_written=4)
WrittenResult(class_name='Cluster', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/cluster'), mode='merge_scoped', predicates=('target."hierarchy_id" = source."hierarchy_id" AND target."id" = source."id" AND target."hierarchy_id" IN (\'tasic_2018_visp_taxonomy\')',), rows_written=138)
WrittenResult(class_name='ClusterHierarchy', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_

In [6]:
# Read back each taxonomy table and verify structural closure and complete colors.
stored_clusters = pl.read_delta(OUTPUT_ROOT / "cluster").filter(
    pl.col("hierarchy_id") == HIERARCHY_ID
)
stored_hierarchy = pl.read_delta(OUTPUT_ROOT / "clusterhierarchy").filter(
    pl.col("id") == HIERARCHY_ID
)
stored_categories = pl.read_delta(OUTPUT_ROOT / "hierarchycategory").filter(
    pl.col("hierarchy_id") == HIERARCHY_ID
)
assert stored_clusters.height == 138
assert stored_clusters["hex_color"].null_count() == 0
assert stored_categories.height == 4
assert stored_hierarchy.height == 1
assert stored_hierarchy.item(0, "root") == ROOT_ID
assert set(stored_hierarchy.item(0, "clusters")) == set(stored_clusters["id"])
assert set(stored_clusters["parent"].drop_nulls()) <= set(stored_clusters["id"])
print("Colored T-type taxonomy verified.")

Colored T-type taxonomy verified.
